# Demo · Los logs de un chatbot

Folio es una librería online que atiende a sus clientes con un chatbot. Ese chatbot funciona con un prompt que el equipo va mejorando, y cada versión tiene un número: en noviembre han usado la `1.1`, la `1.2` y la `1.10`, que es la más reciente.

Queremos saber **qué versión del prompt acierta más**. Los datos llegan de dos sitios distintos:

| Fichero | Quién lo manda | Formato |
| --- | --- | --- |
| `logs_noviembre.jsonl` | Sistemas: el registro de cada conversación | JSONL, con la respuesta del modelo anidada |
| `valoraciones_noviembre.csv` | Soporte: si cada respuesta era correcta y su nota | CSV exportado de Excel en español |

Este notebook lo recorremos juntos en clase. Antes de algunas celdas hay una **pregunta**: pensadla antes de ver el resultado.

In [1]:
import json
from pathlib import Path

import duckdb
import pandas as pd

**Preparar los datos.** Esta celda simula lo que os mandarían los equipos de sistemas y de soporte: crea los ficheros en la misma carpeta que el notebook. Ejecutadla una vez y seguid; no hace falta entenderla.

In [2]:
import json
import random

import pandas as pd

random.seed(2026)
CATEGORIAS = {
    "Pedidos": "¿Dónde está mi pedido?",
    "Devoluciones": "¿Cómo devuelvo un libro?",
    "Pagos": "¿Puedo pagar con Bizum?",
    "Envíos": "¿Cuánto tarda el envío?",
}
ACIERTO = {"1.1": 0.62, "1.2": 0.70, "1.10": 0.84}          # probabilidad de acierto de cada versión del prompt

conversaciones, valoraciones = [], []
for i in range(1, 240 + 1):
    dia = random.randint(1, 30)
    version = "1.1" if dia <= 10 else ("1.2" if dia <= 20 else "1.10")
    categoria = random.choice(list(CATEGORIAS))
    correcta = int(random.random() < ACIERTO[version])
    conversaciones.append({
        "id_conversacion": f"nov-{i:04d}",
        "fecha": f"2026-11-{dia:02d}",
        "categoria": categoria,
        "pregunta": CATEGORIAS[categoria],
        "respuesta": {
            "modelo": random.choice(["modelo_a", "modelo_b"]),
            "version_prompt": version,
            "tokens": random.randint(150, 600),
            "latencia_ms": random.randint(600, 2200),
        },
    })
    nota = random.choice([4, 4.5, 5]) if correcta else random.choice([1, 1.5, 2, 2.5, 3])
    valoraciones.append({"id_conversacion": f"nov-{i:04d}",
                         "revisor": random.choice(["Lucía", "Marcos", "Irene"]),
                         "correcta": correcta, "puntuacion": nota})

with open("logs_noviembre.jsonl", "w", encoding="utf-8") as f:
    for c in conversaciones:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
# soporte exporta desde Excel en formato español: ";" entre columnas y coma decimal
pd.DataFrame(valoraciones).to_csv("valoraciones_noviembre.csv", sep=";", decimal=",", index=False)
print("Datos preparados:", "logs_noviembre.jsonl", "y", "valoraciones_noviembre.csv")

Datos preparados: logs_noviembre.jsonl y valoraciones_noviembre.csv


## 1. Leer el CSV de soporte

**Pregunta.** Si lo leemos sin decirle nada a pandas, ¿cuántas columnas creéis que salen?

In [3]:
pd.read_csv("valoraciones_noviembre.csv").head(3)

,id_conversacion;revisor;correcta;puntuacion
nov-0001;Irene;1;5,0
nov-0002;Lucía;0;3,0
nov-0003;Irene;0;2,5


Una sola, con un nombre larguísimo. Y como la coma decimal parte cada línea en dos, el principio de cada línea se ha ido al índice. pandas espera comas entre columnas y aquí van separadas por punto y coma, porque la coma es la de los decimales. Hay que decírselo con `sep` y `decimal`:

In [4]:
valoraciones = pd.read_csv("valoraciones_noviembre.csv", sep=";", decimal=",")

print(valoraciones.shape)
valoraciones.dtypes

(240, 4)


id_conversacion        str
revisor                str
correcta             int64
puntuacion         float64
dtype: object

`puntuacion` ya es `float64`. Sin `decimal=","`, llegaría como texto y no se podría hacer la media.

## 2. Leer los logs en JSONL

Un JSONL tiene un JSON por línea. Así es una línea por dentro:

In [5]:
with open("logs_noviembre.jsonl", encoding="utf-8") as f:
    print(f.readline())

{"id_conversacion": "nov-0001", "fecha": "2026-11-04", "categoria": "Pagos", "pregunta": "¿Puedo pagar con Bizum?", "respuesta": {"modelo": "modelo_a", "version_prompt": "1.1", "tokens": 264, "latencia_ms": 1830}}



**Pregunta.** La respuesta del modelo va dentro de otro objeto. Si lo leemos directamente, ¿podremos agrupar por versión del prompt?

In [6]:
pd.read_json("logs_noviembre.jsonl", lines=True).head(3)

,id_conversacion,fecha,categoria,pregunta,respuesta
0,nov-0001,2026-11-04,Pagos,¿Puedo pagar con Bizum?,"{'modelo': 'modelo_a', 'version_prompt': '1.1'..."
1,nov-0002,2026-11-14,Envíos,¿Cuánto tarda el envío?,"{'modelo': 'modelo_b', 'version_prompt': '1.2'..."
2,nov-0003,2026-11-04,Pagos,¿Puedo pagar con Bizum?,"{'modelo': 'modelo_b', 'version_prompt': '1.1'..."


No: toda la respuesta queda metida en una columna, como un diccionario. Para tener una columna por campo, leemos cada línea con `json.loads` y aplanamos con `pd.json_normalize`. Con `sep="_"`, las columnas nuevas se llaman `respuesta_modelo`, `respuesta_version_prompt`, etc.

In [7]:
with open("logs_noviembre.jsonl", encoding="utf-8") as f:
    registros = [json.loads(linea) for linea in f]

logs = pd.json_normalize(registros, sep="_")
logs["fecha"] = pd.to_datetime(logs["fecha"])

print(logs.shape)
logs.head(3)

(240, 8)


,id_conversacion,fecha,categoria,pregunta,respuesta_modelo,respuesta_version_prompt,respuesta_tokens,respuesta_latencia_ms
0,nov-0001,2026-11-04,Pagos,¿Puedo pagar con Bizum?,modelo_a,1.1,264,1830
1,nov-0002,2026-11-14,Envíos,¿Cuánto tarda el envío?,modelo_b,1.2,272,605
2,nov-0003,2026-11-04,Pagos,¿Puedo pagar con Bizum?,modelo_b,1.1,155,2003


## 3. Unir logs y valoraciones

Las dos tablas comparten `id_conversacion`. Con `merge` pegamos a cada conversación su valoración. `how="left"` conserva todas las conversaciones aunque alguna no estuviera valorada; si faltara alguna, saldría con huecos.

In [8]:
datos = logs.merge(valoraciones, on="id_conversacion", how="left")

print(datos.shape)
print("Conversaciones sin valorar:", datos["correcta"].isna().sum())

(240, 11)
Conversaciones sin valorar: 0


Primera respuesta con pandas: la tasa de acierto es la media de `correcta`, porque vale 1 o 0.

In [9]:
datos.groupby("respuesta_version_prompt")["correcta"].agg(["mean", "size"]).round(2)

,mean,size
respuesta_version_prompt,,
1.1,0.60,70
1.10,0.81,80
1.2,0.71,90


La `1.10` es la que más acierta. Fijaos en el orden: `1.1`, `1.10`, `1.2`. Es texto, y el texto se ordena letra a letra, como en un diccionario.

## 4. Guardar: CSV contra Parquet

Guardamos la tabla para el resto del equipo en los dos formatos y la volvemos a leer.

**Pregunta.** ¿Qué columna creéis que va a cambiar al volver del CSV?

In [10]:
datos.to_csv("logs_noviembre.csv", index=False)
datos.to_parquet("logs_noviembre.parquet", index=False)

de_csv = pd.read_csv("logs_noviembre.csv")
de_parquet = pd.read_parquet("logs_noviembre.parquet")

print("Versiones desde CSV:    ", sorted(de_csv["respuesta_version_prompt"].unique().tolist()))
print("Versiones desde Parquet:", sorted(de_parquet["respuesta_version_prompt"].unique().tolist()))

Versiones desde CSV:     [1.1, 1.2]
Versiones desde Parquet: ['1.1', '1.10', '1.2']


El CSV no guarda tipos: al leerlo, pandas ve `1.10` y lo convierte en el número `1.1`. La versión nueva se ha mezclado con la antigua. Mirad lo que diría el análisis:

In [11]:
de_csv.groupby("respuesta_version_prompt")["correcta"].agg(["mean", "size"]).round(2)

,mean,size
respuesta_version_prompt,,
1.1,0.71,150
1.2,0.71,90


La mejor versión ha desaparecido y la `1.1` parece mucho mejor de lo que es. No ha saltado ningún error: el informe sería falso y nadie se enteraría.

Hay dos formas de evitarlo:
- Si os obligan a usar CSV, decidle a pandas el tipo: `pd.read_csv("logs_noviembre.csv", dtype={"respuesta_version_prompt": str})`.
- Si el fichero es vuestro, guardadlo en **Parquet**, que conserva el tipo de cada columna. Además, ocupa menos:

In [12]:
for fichero in ["logs_noviembre.jsonl", "logs_noviembre.csv", "logs_noviembre.parquet"]:
    print(f"{fichero:<24} {Path(fichero).stat().st_size:>7} bytes")

logs_noviembre.jsonl       52578 bytes
logs_noviembre.csv         21688 bytes
logs_noviembre.parquet     11488 bytes


Con 240 filas la diferencia es pequeña. Con un millón de filas, la diapositiva de tamaños lo deja claro.

## 5. Preguntar con DuckDB

DuckDB lanza SQL directamente sobre un fichero y devuelve un DataFrame con `.df()`. La misma pregunta de antes, ahora sobre el Parquet:

In [13]:
duckdb.sql("""
SELECT respuesta_version_prompt AS version,
       round(avg(correcta), 2) AS acierto,
       count(*) AS conversaciones
FROM 'logs_noviembre.parquet'
GROUP BY version
ORDER BY acierto DESC
""").df()

,version,acierto,conversaciones
0,1.10,0.81,80
1,1.2,0.71,90
2,1.1,0.60,70


**Pregunta.** Si lanzamos la misma consulta sobre el CSV, ¿DuckDB se libra del problema?

In [14]:
duckdb.sql("""
SELECT respuesta_version_prompt AS version,
       round(avg(correcta), 2) AS acierto,
       count(*) AS conversaciones
FROM 'logs_noviembre.csv'
GROUP BY version
ORDER BY acierto DESC
""").df()

,version,acierto,conversaciones
0,1.1,0.71,150
1,1.2,0.71,90


Tampoco: DuckDB también adivina los tipos al leer un CSV. El problema está en el formato, no en la herramienta.

Una última pregunta: ¿la `1.10` gana en todas las categorías?

In [15]:
duckdb.sql("""
SELECT categoria,
       respuesta_version_prompt AS version,
       round(avg(correcta), 2) AS acierto
FROM 'logs_noviembre.parquet'
GROUP BY categoria, version
ORDER BY categoria, acierto DESC
""").df()

,categoria,version,acierto
0,Devoluciones,1.10,0.89
1,Devoluciones,1.1,0.72
2,Devoluciones,1.2,0.70
3,Envíos,1.10,0.79
4,Envíos,1.2,0.76
5,Envíos,1.1,0.53
6,Pagos,1.10,0.90
7,Pagos,1.2,0.71
8,Pagos,1.1,0.64
9,Pedidos,1.10,0.69


## Qué os lleváis

- **Leer bien es decirle a pandas lo que no puede adivinar:** separador, decimales y tipos.
- **Un JSON anidado se aplana con `json_normalize`** antes de poder agrupar.
- **Guardar en CSV puede cambiar los datos sin avisar.** Para guardar vuestro trabajo, Parquet.
- **DuckDB responde en SQL directamente sobre un fichero**, y devuelve un DataFrame.